# EDA — Prescription panel
Quick exploratory analysis of the aggregated monthly panel produced by `ml/prepare_data.py`.

Run order:
1. `python ml/prepare_data.py`
2. `python ml/build_features.py`
3. open this notebook

In [ ]:
import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt
DATA = os.path.abspath('../data')
panel = pd.read_parquet(os.path.join(DATA, 'monthly_panel.parquet'))
panel.head()

In [ ]:
print('rows:', f'{len(panel):,}')
print('series (region,district,icd):', panel.groupby(['region','district','icdid']).ngroups)
print('regions:', panel['region'].nunique())
print('icd codes:', panel['icdid'].nunique())
print('range:', panel['year_month'].min(), '->', panel['year_month'].max())

In [ ]:
# Country-wide monthly trend
ts = panel.groupby('year_month')['recipe_count'].sum()
plt.figure(figsize=(10, 3))
ts.plot(color='#4f87ff'); plt.title('Total prescriptions per month')
plt.grid(alpha=.2); plt.show()

In [ ]:
# Top 15 ICD codes overall
panel.groupby(['icdid','nozology'])['recipe_count'].sum().sort_values(ascending=False).head(15)

In [ ]:
# Region totals — heatmap (region x ICD chapter)
panel['chapter'] = panel['icdid'].astype(str).str[0]
tab = panel.groupby(['region','chapter'])['recipe_count'].sum().unstack(fill_value=0)
tab = tab.reindex(sorted(tab.index), axis=0)
import seaborn as sns
plt.figure(figsize=(10,8)); sns.heatmap(np.log1p(tab), cmap='turbo'); plt.title('log(1+recipes) by region × ICD chapter'); plt.show()

In [ ]:
# Forecast quality
import json
with open('../models/metadata.json', 'r', encoding='utf-8') as f:
    meta = json.load(f)
pd.DataFrame(meta['metrics']).T